# Parallelized sweep of get_int_aligned_trajectory `max_clear_time` + `refill_percentage`

`get_int_aligned_trajectory` interpolates + physically filters each device
stream before the DTW match. After filtering, if a temporal gap longer than
`max_clear_time` opens up, it re-adds a `refill_percentage` fraction of the
removed points in that gap. These two parameters only matter together, so this
notebook sweeps them as dependent pairs:

| `max_clear_time` | `refill_percentage` |
| --- | --- |
| 30 | 0.3 |
| 30 | 0.1 |
| 10 | 0.3 |
| 10 | 0.1 |

Because these parameters change the INPUT to the DTW (they filter/refill the
interpolated trajectory before matching), every pair needs a full DTW build —
there is no "match once, reprocess many" shortcut like the DTW outlier
time-threshold sweep. We therefore monkeypatch
`emr.get_int_aligned_trajectory` inside each worker so the existing
`emr.ref_dtw_gt_with_ends_general` runs the complete build with the swept
parameters, and fan every `(pair, timeline, section)` combination out across
the joblib loky process pool.

In [ ]:
# for reading and validating data
import emeval.input.spec_details as eisd
import emeval.input.phone_view as eipv
import emeval.input.eval_view as eiev

# Metrics helpers
import emeval.metrics.dist_calculations as emd
import emeval.metrics.reference_trajectory as emr
import emeval.metrics.DTW as dtw

# For computation
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely as shp
import matplotlib.pyplot as plt
import arrow
import os
import copy
import pickle
import itertools
from joblib import Parallel, delayed
import importlib
importlib.reload(emr)

In [ ]:
DATASTORE_LOC = "bin/data"
AUTHOR_EMAIL = "shankari@eecs.berkeley.edu"
sd_la = eisd.FileSpecDetails(DATASTORE_LOC, AUTHOR_EMAIL, "unimodal_trip_car_bike_mtv_la")
sd_sj = eisd.FileSpecDetails(DATASTORE_LOC, AUTHOR_EMAIL, "car_scooter_brex_san_jose")
sd_ucb = eisd.FileSpecDetails(DATASTORE_LOC, AUTHOR_EMAIL, "train_bus_ebike_mtv_ucb")

pv_la = eipv.PhoneView(sd_la)
pv_sj = eipv.PhoneView(sd_sj)
pv_ucb = eipv.PhoneView(sd_ucb)
pvs = [pv_la, pv_sj, pv_ucb]

In [ ]:
def get_reference_trajectory_input_tree(pv):
    ref_tree = {}

    for phone_os, phone_map in pv.map().items():
        for phone_label, phone_detail_map in phone_map.items():
            for (r_idx, r) in enumerate(phone_detail_map["evaluation_ranges"]):
                if r["eval_role_base"] != "accuracy_control":
                    continue
                for (tr_idx, tr) in enumerate(r["evaluation_trip_ranges"]):
                    for (sr_idx, sr) in enumerate(tr["evaluation_section_ranges"]):
                        section_gt_leg = pv.spec_details.get_ground_truth_for_leg(tr["trip_id_base"], sr["trip_id_base"], sr["start_ts"], sr["end_ts"])
                        section_gt_shapes = gpd.GeoSeries(eisd.SpecDetails.get_shapes_for_leg(section_gt_leg))
                        if len(section_gt_shapes) == 1:
                            print("No ground truth route for %s %s, must be polygon, skipping..." % (tr["trip_id_base"], sr["trip_id_base"]))
                            assert section_gt_leg["type"] != "TRAVEL", "For %s, %s, %s, %s, %s found type %s" % (phone_os, phone_label, r_idx, tr_idx, sr_idx, section_gt_leg["type"])
                            continue
                        if len(sr['location_df']) == 0:
                            print("No sensed locations found, role = %s skipping..." % (r["eval_role_base"]))
                            continue

                        print("Processing travel leg %s, %s, %s, %s, %s" %
                              (phone_os, phone_label, r["eval_role_base"], tr["trip_id_base"], sr["trip_id_base"]))
                        sec_name = tr["trip_id_base"] + "/" + sr["trip_id_base"] + "_" + str(r_idx)
                        if sec_name not in ref_tree:
                            ref_tree[sec_name] = {
                                "trip_id": tr["trip_id_base"],
                                "section_id": sr["trip_id_base"],
                                "run": r_idx,
                                "ground_truth": {
                                    "leg": section_gt_leg
                                }
                            }

                        assert sec_name in ref_tree
                        e = ref_tree[sec_name]
                        section_measured_points = sr["location_df"]
                        if "temporal_control" not in e:
                            e["temporal_control"] = {}
                            e["start_ts"] = sr["start_ts"]
                            e["end_ts"] = sr["end_ts"]
                        e["temporal_control"][phone_os] = sr
    return ref_tree

## Build the input trees once, then sweep the `(max_clear_time, refill_percentage)` pairs

The ground-truth / location loading is expensive, so we do it serially once per
timeline and pre-serialize each section's base entry. The sweep itself fans out
every `(pair, timeline, section)` combination across the joblib loky process
pool.

Each worker temporarily replaces `emr.get_int_aligned_trajectory` with a wrapper
that injects the swept `max_clear_time` and `refill_percentage` and then calls
the unmodified `emr.ref_dtw_gt_with_ends_general`, so the full DTW build runs
with the new filter/refill. The patch is process-local (loky uses separate
processes) and is restored after every task.

In [ ]:
# Python 3.11 still has the GIL, so a ThreadPoolExecutor gains nothing for this
# CPU-bound work. Use joblib's process-based parallelism (loky backend) so the
# builders actually run concurrently across cores.
N_JOBS = 8

# max_clear_time and refill_percentage only have an effect together, so we sweep
# them as dependent pairs. The grouping key is a readable label string.
PARAM_NAME = "max_clear_refill"
MAX_CLEAR_VALUES = [30, 10]
REFILL_VALUES = [0.3, 0.1]

# Each entry is (label, kwargs-to-inject-into-get_int_aligned_trajectory).
PARAM_SETTINGS = [
    ("mc%g_rf%g" % (mc, rf), {"max_clear_time": mc, "refill_percentage": rf})
    for mc in MAX_CLEAR_VALUES
    for rf in REFILL_VALUES
]

In [ ]:
# Build the (parameter-independent) input trees once per timeline.
input_trees = {}      # timeline_id -> {sec_name: base_entry}
timeline_tz = {}      # timeline_id -> tz
for pv in pvs:
    input_trees[pv.spec_details.CURR_SPEC_ID] = get_reference_trajectory_input_tree(pv)
    timeline_tz[pv.spec_details.CURR_SPEC_ID] = pv.spec_details.eval_tz

# Pre-serialize every section's base_entry exactly once. base_entry holds the
# raw android/ios location dataframes and is expensive to pickle, so doing it
# here means the same bytes are reused for every parameter task that operates on
# that section instead of joblib re-pickling the live object once per task.
serialized_entries = {}   # timeline_id -> {sec_name: pickled base_entry bytes}
for timeline, tree_in in input_trees.items():
    serialized_entries[timeline] = {
        sec_name: pickle.dumps(base_entry, protocol=pickle.HIGHEST_PROTOCOL)
        for sec_name, base_entry in tree_in.items()
    }

In [ ]:
# The sweep changes the INPUT to the DTW, so unlike the time-threshold sweep we
# cannot cache a single match and reprocess it -- every parameter pair needs a
# full DTW build. ref_dtw_gt_with_ends_general now forwards the filter
# parameters straight through to get_int_aligned_trajectory, so we just pass the
# swept pair in directly.
import traceback


def stats_for_ref(ref_df, e):
    """The same per-reference stats that emr.ref_and_stats computes."""
    emr.speed_acceleration_jerk(ref_df)
    stats = {
        "coverage_density": emr.coverage_density(ref_df, e),
        "coverage_time": emr.coverage_time(ref_df, e),
        "coverage_max_gap": emr.coverage_max_gap(ref_df, e),
        "max_jerk": emr.max_jerk(ref_df, e),
        "max_acceleration": emr.max_acceleration(ref_df, e),
        "max_speed": emr.max_speed(ref_df, e),
        "median_jerk": emr.median_jerk(ref_df, e),
        "mean_median_jerk_ratio": emr.mean_median_jerk_ratio(ref_df, e),
    }
    try:
        stats["gt_error"] = emd.dist_using_projection_adjusted(ref_df, e["ground_truth"]["linestring"])
    except Exception as exp_gt:
        print("Found exception %s while computing gt_error" % exp_gt)
        stats["gt_error"] = np.nan
    return stats


def process_for_params(param_label, param_kwargs, timeline, sec_name, base_entry_bytes, tz):
    e = pickle.loads(base_entry_bytes)
    try:
        emr.fill_gt_linestring(e)
        ref_df = emr.ref_dtw_gt_with_ends_general(e, tz=tz, **param_kwargs)
        if ref_df is None or len(ref_df) == 0:
            return {"__error__": "empty ref_df", PARAM_NAME: param_label,
                    "timeline": timeline, "sec_name": sec_name}
        stats = stats_for_ref(ref_df, e)
    except Exception as exp:
        # Return the failure (with traceback) instead of None so we can see why
        # a task failed -- prints inside loky workers are easy to miss.
        return {"__error__": "%s: %s" % (type(exp).__name__, exp),
                "__traceback__": traceback.format_exc(),
                PARAM_NAME: param_label, "timeline": timeline, "sec_name": sec_name}

    row = {
        PARAM_NAME: param_label,
        "max_clear_time": param_kwargs["max_clear_time"],
        "refill_percentage": param_kwargs["refill_percentage"],
        "timeline": timeline,
        "sec_name": sec_name,
        "n_points": len(ref_df),
    }
    row.update(stats)
    return row


# Fan out every (parameter pair, timeline, section) combination.
sweep_tasks = []
for param_label, param_kwargs in PARAM_SETTINGS:
    for timeline, tree_in in input_trees.items():
        for sec_name in tree_in:
            sweep_tasks.append((param_label, param_kwargs, timeline, sec_name,
                                serialized_entries[timeline][sec_name],
                                timeline_tz[timeline]))

print("Processing %d %s x segment combinations on %d processes" %
      (len(sweep_tasks), PARAM_NAME, N_JOBS))
sweep_results = Parallel(n_jobs=N_JOBS, verbose=10)(
    delayed(process_for_params)(*t) for t in sweep_tasks)

# Split successes from failures so an all-failed run reports *why* instead of
# producing an empty DataFrame that blows up on the later groupby.
sweep_rows = [r for r in sweep_results if r is not None and "__error__" not in r]
sweep_errors = [r for r in sweep_results if r is not None and "__error__" in r]
sweep_df = pd.DataFrame(sweep_rows)
print("Collected %d of %d reference stats (%d failed)" %
      (len(sweep_df), len(sweep_tasks), len(sweep_errors)))
if len(sweep_df) == 0 and sweep_errors:
    print("\nAll tasks failed. First failure traceback:\n")
    print(sweep_errors[0].get("__traceback__", sweep_errors[0]["__error__"]))
sweep_df.head()

NameError: name 'PARAM_SETTINGS' is not defined

In [ ]:
# Persist the raw per-section sweep so it can be reloaded without rebuilding.
sweep_df.to_csv("get_int_aligned_max_clear_refill_sweep.csv", index=False)

## Aggregate across sections per `(max_clear_time, refill_percentage)` pair

In [ ]:
assert len(sweep_df) > 0, (
    "sweep_df is empty -- every task failed, so there is nothing to aggregate. "
    "Check the failure traceback printed by the sweep cell above.")

agg_metrics = [m for m in ["gt_error", "coverage_density", "coverage_time",
                           "coverage_max_gap", "max_jerk", "max_acceleration",
                           "max_speed", "median_jerk", "mean_median_jerk_ratio",
                           "n_points"]
               if m in sweep_df.columns]
param_summary = sweep_df.groupby(PARAM_NAME)[agg_metrics].mean()
param_summary["n_sections"] = sweep_df.groupby(PARAM_NAME).size()
param_summary

In [ ]:
# Bar plot of the mean ground-truth error for each parameter pair; the pair that
# minimizes gt_error is the best candidate.
if "gt_error" in param_summary.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(param_summary.index.astype(str), param_summary["gt_error"])
    best = param_summary["gt_error"].idxmin()
    ax.set_xlabel("max_clear_time / refill_percentage")
    ax.set_ylabel("mean ground-truth error (m)")
    ax.set_title("DTW reference accuracy vs. max_clear/refill (best: %s)" % best)
    fig

In [ ]:
# Plot box plots of median jerk for every parameter pair to see how much noise
# remains in the trajectories at different settings.
if "median_jerk" in sweep_df.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    # We drop any NaNs/Infs to ensure the boxplot renders properly
    clean_df = sweep_df[np.isfinite(sweep_df["median_jerk"])]
    clean_df.boxplot(column="median_jerk", by=PARAM_NAME, ax=ax, grid=True)
    ax.set_xlabel("max_clear_time / refill_percentage")
    ax.set_ylabel("Median Jerk ($m/s^3$)")
    ax.set_title("Distribution of Median Jerk by max_clear/refill")
    plt.suptitle("")  # Clear the automatic pandas subtitle
    fig